In [2]:
import os, pickle

from physics.simulation import mcfm, msq
from physics.hzz import zz4l
from datasets import balanced
from models import carl

import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import hist
import vector

import torch
from torch.utils.data import Dataset
import lightning

In [3]:
torch.set_float32_matmul_precision('high')

matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,  # Don't override with default matplotlib fonts
    "pgf.preamble": "", 
})


In [ ]:
OUTPUT_DIR = 'run/h2l2v/carl/'
SCALER_FILE_X = 'scaler.pkl'
SAMPLE_DIR = 'data/'

TRAIN_EPOCH = '08'
VAL_LOSS = '0.69'
VERSION = '1'

LOGS_DIR = f'{OUTPUT_DIR}/lightning_logs/version_{VERSION}'
CHECKPOINT_DIR = f'{LOGS_DIR}/checkpoints/'
CHECKPOINT = f'epoch={TRAIN_EPOCH}-val_loss={VAL_LOSS}.ckpt'

BATCH_SIZE = 4096

In [7]:
FEATURES = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy', 'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy', 'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy', 'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
FEATURES = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]

In [8]:
with open(f'{OUTPUT_DIR}/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_numerator_train.pkl', 'rb') as f:
    events_numerator_train = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_denominator_train.pkl', 'rb') as f:
    events_denominator_train = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_numerator_val.pkl', 'rb') as f:
    events_numerator_val = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_denominator_val.pkl', 'rb') as f:
    events_denominator_val = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_numerator_test.pkl', 'rb') as f:
    events_numerator_test = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_denominator_test.pkl', 'rb') as f:
    events_denominator_test = pickle.load(f)

training_data = balanced.BalancedDataset(events_numerator_train, events_denominator_train, FEATURES, scaler=scaler)
validation_data = balanced.BalancedDataset(events_numerator_val, events_denominator_val, FEATURES, scaler=scaler)
testing_data = balanced.BalancedDataset(events_numerator_test, events_denominator_test, FEATURES, scaler=scaler)

In [9]:
loaded_model = carl.CARL.load_from_checkpoint(os.path.join(CHECKPOINT_DIR, CHECKPOINT))

dl_train = torch.utils.data.DataLoader(training_data, batch_size=BATCH_SIZE)
dl_val = torch.utils.data.DataLoader(validation_data, batch_size=BATCH_SIZE)
dl_test = torch.utils.data.DataLoader(testing_data, batch_size=BATCH_SIZE)

trainer = lightning.Trainer(accelerator='gpu', devices=1)

# predictions_train = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_train]), axis=0).detach().numpy()
predictions_val_batches = trainer.predict(loaded_model, dataloaders=[dl_val])
predictions_test_batches = trainer.predict(loaded_model, dataloaders=[dl_test])

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIB

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [14]:
predictions_val = torch.cat(predictions_val_batches).detach().numpy()
predictions_test = torch.cat(predictions_test_batches).detach().numpy()

# targets_train = training_data.s
targets_val = validation_data.s
targets_test = testing_data.s

In [15]:
s_nbins = 50
s_min, s_max = 0.0, 1.0
s_step = (s_max - s_min)/s_nbins

In [16]:
p = predictions_test
t = targets_test

pred_binned = [p[(p > s_min+s_step*i) & (p <= s_min+s_step*(i+1))] for i in range(s_nbins)]
targets_binned = [t[(p > s_min+s_step*i) & (p <= s_min+s_step*(i+1))] for i in range(s_nbins)]

sig_per_bin = np.array([np.sum(targets_binned[i]==1.0) for i in range(s_nbins)])
bkg_per_bin = np.array([np.sum(targets_binned[i]==0.0) for i in range(s_nbins)])

s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)

sig_err = np.array([np.std(targets_binned[i]==1.0) for i in range(s_nbins)])
bkg_err = np.array([np.std(targets_binned[i]==0.0) for i in range(s_nbins)])
s_err = np.sqrt((sig_err * bkg_per_bin/(sig_per_bin + bkg_per_bin)**2)**2 + (-bkg_err*sig_per_bin/(sig_per_bin + bkg_per_bin)**2)**2)

s_centers = [s_min+(i+1/2)*s_step for i in range(s_nbins)]


/tmp/ipykernel_3038117/1948266598.py:10: RuntimeWarning: invalid value encountered in divide
  s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [17]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,5), sharex=True)
plt.subplots_adjust(wspace=0, hspace=0)
ax1.set_aspect('equal', adjustable='box')

ax1.errorbar(s_centers, s_centers, color='black', linestyle='--', label='$\\mathrm{MC}$')
ax1.errorbar(s_centers, s_true, yerr=s_err, color='royalblue', marker='+', markersize=4, linestyle='none', label='$\\mathrm{NSBI}$')

ax1.set_ylim(s_min, s_max)
ax1.set_ylabel('$\\mathrm{MC\\ estimate}\\ \\frac{p_{q\\bar{q}}(x)}{ p_{q\\bar{q}}(x) + p_{gg}(x) }$', fontsize=15)

ax1.legend(frameon=False, fontsize=12)

ax2.errorbar(s_centers, np.array(s_centers)-np.array(s_centers), color='black', linestyle='--')
ax2.errorbar(s_centers, np.array(s_true)-np.array(s_centers), yerr=0., color='royalblue', marker='+', markersize=4, linestyle='none')

ax2.set_xlim(s_min, s_max)
ax2.set_xlabel('$\\mathrm{NSBI\\ estimate}\\ \\hat{s}(x)$', fontsize=15)
ax2.set_ylabel('$\\mathrm{Residual}$', fontsize=15)
ax2.set_ylim(-0.075, 0.075)
ax2.yaxis.set_ticks([-0.05, 0.05])

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)

plt.tight_layout()
plt.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

plt.savefig('carl_calibration.pdf', bbox_inches='tight')
plt.close()

In [39]:
predictions_denominator_test = loaded_model(torch.tensor(scaler.transform(events_denominator_test.kinematics[FEATURES].to_numpy()), dtype=torch.float32)).detach().numpy().ravel()

In [40]:
# m4l_numerator = events_numerator_test.kinematics['4l_mass']
# m4l_denominator = events_denominator_test.kinematics['4l_mass']
# xobs_numerator = m4l_numerator
# xobs_denominator = m4l_denominator
# bins = 41
# bounds = [180,1000]

mtzz_numerator = events_numerator_test.kinematics['zz_mt']
mtzz_denominator = events_denominator_test.kinematics['zz_mt']
xobs_numerator = mtzz_numerator
xobs_denominator = mtzz_denominator
bins = 31
bounds = [270,1200]

In [41]:

h_true = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_true.fill(xobs_numerator, weight=events_numerator_test.probabilities)

h_gg = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_gg.fill(xobs_denominator, weight=events_denominator_test.probabilities)

h_reweight = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_reweight.fill(xobs_denominator, weight=(predictions_denominator_test / (1 - predictions_denominator_test))* events_denominator_test.probabilities)

fig, (ax1, ax2, ax3) = plt.subplots(3,1,gridspec_kw={'height_ratios': [2, 1, 1]},figsize=(6,8), layout='constrained', sharex=True)

h_true.plot(ax=ax1,color='royalblue', label='$q\\bar{q} \\to ZZ$')
h_gg.plot(ax=ax1,color='black', linestyle='--', label='$gg(\\to h^{\\ast}) \\to ZZ$')
h_reweight.plot(ax=ax1,color='red', label='ggZZ $\\rightarrow$ qqZZ (estimated)')

ax1.set_xlabel('')
ax1.set_ylabel('probability', fontsize=12)

ax1.set_xticklabels([])
ax1.tick_params(labelsize=12)
ax1.xaxis.set_tick_params(which='minor', labelleft=False, labelright=False)

ax1.set_yscale('log')
ax1.legend(frameon=False, fontsize=12)

ax2.errorbar(h_true.axes[0].centers, h_true.values()/h_gg.values(), yerr=np.sqrt((np.sqrt(h_true.variances())/h_gg.values())**2 + (-np.sqrt(h_gg.variances())*h_true.values()/h_gg.values()**2)**2), color='royalblue', drawstyle='steps-mid', label=f'Truth qqZZ', alpha=0.8)
ax2.errorbar(h_reweight.axes[0].centers, h_reweight.values()/h_gg.values(), yerr=np.sqrt((np.sqrt(h_reweight.variances())/h_gg.values())**2 + (-np.sqrt(h_gg.variances())*h_reweight.values()/h_gg.values()**2)**2), color='r', drawstyle='steps-mid', label=f'CARL ggZZ->qqZZ')

ax2.tick_params(labelsize=12)

ax2.set_ylabel('$p_{qq}/p_{gg}$', fontsize=12)
ax2.set_ylim(0.6,2.4)
ax2.xaxis.set_tick_params(which='minor', labelleft=False, labelright=False)

ax2.set_xlabel('')
ax2.set_xticklabels([])

ax3.errorbar(h_true.axes[0].centers, h_true.values()/h_true.values(), yerr=np.sqrt(h_true.variances())/h_true.values(), color='royalblue', drawstyle='steps-mid', label=f'Truth qqZZ', alpha=0.8)
# ax3.errorbar(h_reweight.axes[0].centers, h_reweight.values()/h_true.values(), yerr=np.sqrt((np.sqrt(h_reweight.variances())/h_true.values())**2 + (-np.sqrt(h_true.variances())*h_reweight.values()/h_true.values()**2)**2), color='r', drawstyle='steps-mid', label=f'CARL ggZZ->qqZZ')

ax3.tick_params(labelsize=12)

ax3.set_xlabel('$m_{ZZ}\\ \\mathrm{[GeV]}$', fontsize=12)
ax3.set_ylabel('$\\mathrm{NSBI}/\\mathrm{MC}$', fontsize=12)
ax3.set_ylim(0.85,1.25)

plt.tight_layout()
plt.subplots_adjust(hspace=0)

plt.savefig('carl_reweight.pdf', bbox_inches='tight')

/tmp/ipykernel_3038117/3306799897.py:47: UserWarning: The figure layout has changed to tight
  plt.tight_layout()
